# Databricks Workspace & Clusters

Before writing any data pipelines, you need to understand the environment you're working in. The **Databricks workspace** is where all your work lives — notebooks, jobs, data, and code. The **cluster** is the compute that runs it.

In this notebook we'll cover:
- How a Databricks workspace is structured
- The two cluster types and when to use each
- How to configure a cluster (runtime, nodes, auto-scaling, auto-termination)
- Cluster access modes and their Unity Catalog implications
- Cluster pools for faster startup

## Workspace Structure

A **workspace** is an isolated Databricks environment — one per team, project, or environment (dev/prod). Inside a workspace you have:

```
Workspace
├── Notebooks          — interactive notebooks (Python, SQL, Scala, R)
├── Repos              — Git-backed folders; sync with GitHub/GitLab/Bitbucket
├── Files              — arbitrary files (configs, scripts, small datasets)
├── Clusters           — compute definitions
├── Jobs               — automated pipeline schedules
├── SQL Warehouses     — dedicated compute for Databricks SQL / BI tools
├── Delta Live Tables  — declarative pipeline definitions
└── Catalog (Unity)    — data governance layer (catalogs, schemas, tables)
```

### DBFS — Databricks File System

DBFS is a distributed file system mounted on every cluster. It abstracts the underlying cloud object storage (S3/ADLS/GCS) with a familiar path syntax:

```
dbfs:/user/hive/warehouse/   ← default location for managed Delta tables
dbfs:/FileStore/             ← files uploaded via the UI
dbfs:/mnt/my-bucket/         ← mount point for an external S3/ADLS bucket
```

> With Unity Catalog, you should prefer **external locations** and **volumes** over DBFS mounts — they offer proper access control and auditing. DBFS is still present but is a legacy pattern.

## Two Cluster Types

Every piece of Spark code in Databricks runs on a cluster. There are two types:

| | All-Purpose Cluster | Job Cluster |
|---|---|---|
| **Created by** | Manually in the UI or API | Automatically when a job runs |
| **Lifecycle** | Persistent — stays running until terminated | Ephemeral — starts fresh, deleted when job ends |
| **Use case** | Interactive development, notebooks, exploration | Automated production jobs |
| **Cost** | Higher (DBU rate, often idle) | Lower (only runs during job, spot-instance friendly) |
| **State** | Shared between users on the cluster | Isolated to the single job run |
| **Restart** | Can be restarted and reused | Cannot be restarted — a new one is created each run |

**Rule of thumb:** Develop on an all-purpose cluster. Run in production with job clusters.

A single Databricks **job** can define multiple **tasks**, each using its own job cluster — or all tasks can share one cluster. Sharing a cluster across tasks saves startup time but means tasks aren't fully isolated.

## Cluster Configuration

### Databricks Runtime (DBR)

The DBR version determines which version of Spark, Python, and pre-installed libraries you get. Always pin to a specific version in production — avoid `Latest`.

| Runtime | What's included | When to use |
|---------|----------------|-------------|
| **Databricks Runtime X.Y** | Spark, Delta Lake, Python, Scala | Standard data engineering |
| **Databricks Runtime X.Y ML** | + MLflow, TensorFlow, PyTorch, scikit-learn | ML training/inference |
| **Databricks Runtime X.Y Photon** | + Photon (C++ vectorized engine) | Heavy SQL / BI workloads needing max performance |

LTS (Long-Term Support) versions are prefixed with `LTS` and receive bug/security fixes for two years — use these for stable production clusters.

### Node Types

A cluster has one **driver node** and zero or more **worker nodes**:

- **Driver** — coordinates the Spark application, hosts the SparkContext, runs the notebook's Python process
- **Workers** — execute Spark tasks in parallel; each worker has one or more executors

A **Single Node** cluster has only a driver (no workers). Spark still works, running locally — useful for small datasets, development, or tasks that don't parallelize (e.g., single-threaded ML training on a GPU node).

### Auto-Scaling

With auto-scaling enabled, Databricks adds or removes worker nodes based on workload:

```
Min workers: 2    Max workers: 8
  → Starts at min, scales up under load, scales back down when idle
```

- Good for workloads with variable concurrency or unpredictable data volumes
- Not ideal for streaming jobs — the continuous nature makes scaling decisions less predictable
- Job clusters often use a fixed size for deterministic cost and runtime

### Auto-Termination

All-purpose clusters should have auto-termination set (e.g., 30–60 minutes of inactivity). Without it, an idle cluster keeps running and accumulating cost. Job clusters terminate automatically — auto-termination is not relevant for them.

## Cluster Access Modes

Access mode controls which users can run code on the cluster and what isolation guarantees they have. This is especially important with Unity Catalog.

| Access Mode | Multi-user? | Unity Catalog | Use case |
|-------------|-------------|---------------|----------|
| **Single User** | No — only the assigned user | Full support | Production jobs, secure workloads |
| **Shared** | Yes — multiple users, isolated | Full support | Shared dev clusters, cost optimisation |
| **No Isolation Shared** *(legacy)* | Yes — no user isolation | Not supported | Legacy; avoid for new workloads |

Key points:
- **Unity Catalog requires Single User or Shared mode** — No Isolation Shared clusters cannot enforce row/column-level security
- Shared mode runs each user's code in an isolated process — one user's crash doesn't affect others
- Job clusters always run in Single User mode (the job's service principal is the single user)

> **Exam tip:** If a question mentions Unity Catalog + cluster configuration, the answer will never be "No Isolation Shared".

## Cluster Pools

Starting a cluster takes 3–5 minutes because cloud VMs need to be provisioned and the Databricks runtime needs to install. **Cluster pools** eliminate most of this wait by keeping a set of VMs pre-warmed and idle.

```
Pool: "Engineering Pool"
  Min idle instances: 5
  Max capacity: 50
  Idle instance timeout: 60 min
  ↓
Cluster A attaches to pool → instant startup (VM already running)
Cluster B attaches to pool → instant startup
```

- Pool VMs are billed at the VM rate (no DBU) while idle — cost vs speed tradeoff
- Multiple clusters can draw from the same pool
- Pools are most valuable for job clusters that run frequently — avoids paying 3–5 min startup overhead repeatedly

### Cluster Policies

Admins can define **cluster policies** — JSON rules that restrict what users can configure when creating clusters:

```json
{
  "autotermination_minutes": { "type": "fixed", "value": 60 },
  "num_workers":             { "type": "range", "minValue": 1, "maxValue": 10 }
}
```

Policies prevent cost overruns (e.g., accidentally creating a 50-node cluster) and enforce organisational standards.

## Hands-On: Inspecting the Cluster

When a notebook is attached to a cluster, `spark` and `sc` (SparkContext) are pre-available. You can inspect cluster properties at runtime.

In [ ]:
# Spark and runtime version
print(f"Spark version : {spark.version}")
print(f"Python version: {spark.conf.get('spark.databricks.clusterUsageTags.sparkVersion', 'n/a')}")

In [ ]:
# Cluster size visible through SparkContext
print(f"App name        : {sc.appName}")
print(f"Master          : {sc.master}")
print(f"Default parallelism (cores): {sc.defaultParallelism}")

In [ ]:
# Spark config — inspect key settings
keys = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.databricks.delta.preview.enabled",
]
for k in keys:
    print(f"{k}: {spark.conf.get(k, 'not set')}")

In [ ]:
# Tune shuffle partitions for the cluster size
# Default is 200 — too high for small clusters, too low for very large ones
# A common rule: 2-4 partitions per CPU core
num_cores = sc.defaultParallelism
recommended = num_cores * 2

spark.conf.set("spark.sql.shuffle.partitions", recommended)
print(f"Set shuffle partitions to {recommended} (cores={num_cores})")

In [ ]:
# dbutils — the Databricks utility library
# Available in notebooks; not available in plain Python scripts outside Databricks

# List the root of DBFS
dbutils.fs.ls("dbfs:/")

In [ ]:
# dbutils.widgets — pass parameters into a notebook (used in Jobs)
dbutils.widgets.text("env", "dev", "Environment")
env = dbutils.widgets.get("env")
print(f"Running in environment: {env}")

In [ ]:
# dbutils.notebook.exit — return a value from a notebook
# Used when one Job task calls another notebook as a subtask
# Uncomment to exit:
# dbutils.notebook.exit("success")

## Summary

- A **workspace** holds notebooks, repos, clusters, jobs, SQL warehouses, and Unity Catalog — all in one environment.
- **All-purpose clusters** are persistent, used for interactive development. **Job clusters** are ephemeral, created per job run — use them in production to save cost.
- **DBR version** controls Spark, Python, and library versions — pin to a specific LTS version for production. DBR ML adds ML frameworks; DBR Photon adds a vectorized C++ engine for SQL-heavy workloads.
- **Auto-scaling** adjusts worker count to match load; **auto-termination** stops idle all-purpose clusters automatically.
- **Access mode** determines user isolation and Unity Catalog support. Single User and Shared modes support Unity Catalog; No Isolation Shared does not.
- **Cluster pools** keep VMs pre-warmed to eliminate the 3–5 minute startup delay. **Cluster policies** let admins enforce cost and configuration guardrails.
- `dbutils` is the notebook utility library — use it to interact with DBFS, pass widget parameters, and control notebook execution flow.

Next: `03-delta-lake-on-databricks.ipynb` — Delta Lake's transaction log, schema enforcement, and Databricks-specific optimizations.